# Recess Ready: Philadelphia

**Question:** each evening, can we predict whether tomorrow will be unsafe for outdoor recess at Philadelphia schools, and **why** (heat, cold, bad air, ozone smog, flash-flood rain, dangerous wind, thunderstorms, heavy snow, or ice)?

*Prototype, not official guidance. Always follow official National Weather Service alerts and your school district.*

## Step 1: Download Philadelphia's daily weather and air quality

**How much history?** Weather history goes back decades, so we take **30+ years of weather (1995 onward)**. That gives the model many unusual winters and summers to learn from. Open-Meteo's air-quality history only starts in **August 2022**, so air values are empty before that.

We ask Open-Meteo (free, no API key) for one row per day:
- **Weather** (archive API): hottest and coldest "feels like" temperature, rain, snow, strongest wind gust, **wind speed and direction**, and a **weather code**. The weather code is a number that says the worst weather of the day, for example 95 = thunderstorm and 66 = freezing rain.
- **Air quality** (air-quality API), every hour, turned into one number per day:
  - **PM2.5** and **PM10**: tiny and small particles (smoke, dust, exhaust)
  - **Ozone**: smog that forms on hot, sunny days
  - **Nitrogen dioxide (NO₂)**: mostly from cars, trucks and power plants
  - **US AQI**: the official EPA Air Quality Index (0-500) that combines all of these. For Philadelphia we also get each pollutant's own AQI, so we can say which one is the **main pollutant** each day.

We write this as two small **functions** (`download` and `daily_table`) because later we reuse them for 35 places around Philadelphia, and we want every place to be handled exactly the same way.

Note: Open-Meteo's air quality numbers are **estimates from a computer model (CAMS)**, not readings from a sensor. We say this in our write-up.

**Saved data:** the first run downloads everything and saves it as CSV files in `data/`. Later runs read those files instead, which is faster and works even if Open-Meteo is slow or down. To download fresh data, delete the files in `data/`.

In [ ]:
import pandas as pd
import requests
import time
from pathlib import Path

# Downloaded data is saved here, so later runs (and judges) don't need to download it again
DATA_DIR = Path("data") if Path("data").exists() else Path("../data")
DATA_DIR.mkdir(exist_ok=True)

LAT, LON = 39.95, -75.16            # Philadelphia (Center City)
WEATHER_START = "1995-01-01"      # 30+ years of weather history
START = AIR_START = "2022-08-01"   # air-quality history starts here
END = "2026-11-12"

WEATHER_URL = "https://archive-api.open-meteo.com/v1/archive"
AIR_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
AIR_VARS = "pm2_5,ozone,pm10,nitrogen_dioxide,us_aqi"
AQI_PARTS = ("us_aqi_pm2_5,us_aqi_pm10,us_aqi_nitrogen_dioxide,us_aqi_ozone,"
             "us_aqi_sulphur_dioxide,us_aqi_carbon_monoxide")   # each pollutant's own AQI
DAILY_VARS = ("apparent_temperature_max,apparent_temperature_min,precipitation_sum,snowfall_sum,"
              "wind_gusts_10m_max,weather_code,wind_speed_10m_max,wind_direction_10m_dominant")

def get_json(url, params, tries=4):
    # Ask a website for data; if it's slow or busy, wait a bit and try again
    for attempt in range(tries):
        try:
            r = requests.get(url, params=params, timeout=120)
            r.raise_for_status()
            return r.json()
        except Exception as err:
            if attempt == tries - 1:
                raise
            print(f"Download problem ({type(err).__name__}), trying again in {10 * (attempt + 1)} seconds...")
            time.sleep(10 * (attempt + 1))

def download(lats, lons, start, end, tz="America/New_York", air_vars=AIR_VARS, batch=5, air_start=None):
    # Ask Open-Meteo for many places, a few at a time (big requests can time out)
    weather, air = [], []
    for i in range(0, len(lats), batch):
        where = {"latitude": ",".join(str(x) for x in lats[i:i + batch]),
                 "longitude": ",".join(str(x) for x in lons[i:i + batch]),
                 "start_date": start, "end_date": end, "timezone": tz}
        w = get_json(WEATHER_URL, {**where, "daily": DAILY_VARS})
        a = get_json(AIR_URL, {**where, "hourly": air_vars, "start_date": air_start or start})
        if isinstance(w, dict):       # one place comes back as one answer; make it a list
            w, a = [w], [a]
        weather += w
        air += a
    return weather, air

def cached(name, make_table):
    # Use the saved CSV in data/ if we have one; otherwise download, then save it
    path = DATA_DIR / f"{name}.csv"
    if path.exists():
        print(f"Using saved data: {path}")
        return pd.read_csv(path, parse_dates=["time"])
    table = make_table()
    table.to_csv(path, index=False)
    print(f"Downloaded and saved: {path}")
    return table

def daily_table(weather, air):
    # Turn one place's answers into a table with one row per day
    w = pd.DataFrame(weather["daily"])
    w["time"] = pd.to_datetime(w["time"])
    a = pd.DataFrame(air["hourly"]).set_index("time")
    a.index = pd.to_datetime(a.index)
    air_daily = pd.DataFrame({
        "pm2_5": a["pm2_5"].resample("D").mean(),                     # average of the 24 hours
        "ozone_8h": a["ozone"].rolling(8).mean().resample("D").max(),  # worst 8-hour average (EPA style)
        "pm10": a["pm10"].resample("D").mean(),
        "nitrogen_dioxide": a["nitrogen_dioxide"].resample("D").mean(),
        "us_aqi": a["us_aqi"].resample("D").max(),                     # the day's worst AQI
    })
    parts = [c for c in a.columns if c.startswith("us_aqi_")]
    if parts:   # which pollutant had the highest AQI that day?
        worst = a[parts].resample("D").max().dropna(how="all")   # skip days with no AQI data at all
        air_daily["main_pollutant"] = worst.idxmax(axis=1).str.replace("us_aqi_", "")
    return w.merge(air_daily.reset_index(), on="time", how="left")   # keep weather days even with no air data

def philly_table():
    weather, air = download([LAT], [LON], WEATHER_START, END, air_vars=AIR_VARS + "," + AQI_PARTS, air_start=AIR_START)
    return daily_table(weather[0], air[0])

df = cached("philly_daily_1995", philly_table)
print(df.shape)
df.head()

## Step 2: How many unsafe *school* days are there, and why?

A day is **unsafe** if any one of these happens. ⚠️ **Team to-do:** look up and cite a real source for each threshold (e.g. NWS, US EPA, or our school district's recess policy), and change the number if the source says something different.

| Warning | Rule |
|---|---|
| Too hot | "feels like" max ≥ 32 °C (≈90 °F, NWS "extreme caution") |
| Too cold | "feels like" min ≤ −7 °C (≈20 °F) |
| Bad air (smoke/particles) | daily PM2.5 > 35.4 µg/m³ (US EPA "Unhealthy for Sensitive Groups") |
| Ozone smog | worst 8-hour ozone > 137 µg/m³ (≈70 ppb, US EPA standard) |
| Flash flood / heavy rain | ≥ 25 mm (≈1 inch) of rain in the day |
| Dangerous wind (hurricanes, storms) | gusts ≥ 74 km/h (≈46 mph, NWS wind advisory level) |
| Heavy snow | ≥ 5 cm (≈2 inches) of snow |

**Dropped after our first run: thunderstorms and ice.** We first included them, but Open-Meteo's history recorded **0** thunderstorm or freezing-rain days in Philadelphia, which can't be true. Its weather history comes from a weather model, not from people watching the sky, so it almost never labels them. A model can't learn a danger its data never shows, so we removed them. In Step 9b we check real airport-station records to see how many we're missing.

**Left out on purpose:** tornadoes (not in our data, very rare in Philadelphia; "dangerous wind" covers the storms that bring them), earthquakes (nobody can predict them, and they aren't climate), and UV/sunburn (not in our weather data).

We only count **school days**: weekdays from September to June (no recess in July and August). We ignore holidays for now.

**Why this check comes first:** if a warning almost never happens, the model can't learn it. We'd drop it or change the threshold now, not on Sunday.

Air dangers before August 2022 are **unknown** (empty), not "safe", because there's no air data yet.

In [ ]:
WEATHER_HAZARDS = ["too_hot", "too_cold", "heavy_rain", "high_wind", "heavy_snow"]
AIR_HAZARDS = ["bad_air", "ozone_smog"]
hazards = WEATHER_HAZARDS + AIR_HAZARDS

def add_hazards(t):
    # Mark each day True/False for each danger, using the rules in the table above
    t["too_hot"] = t["apparent_temperature_max"] >= 32
    t["too_cold"] = t["apparent_temperature_min"] <= -7
    t["heavy_rain"] = t["precipitation_sum"] >= 25
    t["high_wind"] = t["wind_gusts_10m_max"] >= 74
    t["heavy_snow"] = t["snowfall_sum"] >= 5
    # Air dangers stay empty (unknown) on days with no air data
    t["bad_air"] = (t["pm2_5"] > 35.4).where(t["pm2_5"].notna())
    t["ozone_smog"] = (t["ozone_8h"] > 137).where(t["ozone_8h"].notna())
    t["unsafe"] = t[hazards].fillna(False).astype(bool).any(axis=1)   # True if ANY known danger happens
    return t

def is_school_day(dates):
    # Weekdays from September to June (we ignore holidays for now)
    return (dates.dt.day_of_week < 5) & ~dates.dt.month.isin([7, 8])

df = add_hazards(df)
school = df[is_school_day(df["time"])].copy()
school["year"] = school["time"].dt.year

print("School days:", len(school), "from", school["time"].min().date(), "to", school["time"].max().date())
print("Unsafe school days:", school["unsafe"].sum())
print()
print("Unsafe school days per year, by reason (air dangers only counted from 2022):")
print(school.groupby("year")[hazards + ["unsafe"]].sum().astype(int).to_string())

## Step 3: Turn each day into a question about *tomorrow*

Right now each row is one day. We want: "using **today's** weather and air, predict **tomorrow's** dangers."

- `features` = what we know today (the model's inputs).
- `shift(-1)` moves tomorrow's answers up one row, so each row holds today's inputs **and** tomorrow's answers side by side.
- We keep only rows where **tomorrow is a school day**.

**Leakage check:** the inputs are only from today. Nothing from tomorrow goes into `features`.

**Wind direction trick:** direction is an angle, so 359° and 1° are almost the same direction even though the numbers are far apart. We turn the angle into two numbers, `sin` and `cos`, which are close together for directions that are close together. That way the model can learn things like "wind from the southwest brings smog".

**Two groups of inputs:**
- `WEATHER_FEATURES`: today's weather, wind and season, plus **trends**: yesterday's feels-like temperatures, how much the low changed since yesterday, 3-day averages, 3-day rain total and yesterday's gusts. These help the model spot a cold spell or storm building up. They only use **past** days, so nothing leaks from tomorrow. These exist for all 30+ years.
- `AIR_FEATURES`: today's PM2.5, ozone, PM10, nitrogen dioxide and US AQI. These only exist from August 2022.

In [ ]:
df = df.sort_values("time").reset_index(drop=True)

import numpy as np

def add_features(t):
    # t must be sorted by date (one place at a time), because some inputs look at earlier days
    t["month"] = t["time"].dt.month
    angle = np.radians(t["wind_direction_10m_dominant"])
    t["wind_dir_sin"] = np.sin(angle)
    t["wind_dir_cos"] = np.cos(angle)
    # Season as a smooth circle (Dec 31 and Jan 1 end up next to each other)
    day = 2 * np.pi * t["time"].dt.day_of_year / 365.25
    t["season_sin"], t["season_cos"] = np.sin(day), np.cos(day)
    # Trends: yesterday, the change since yesterday, and 3-day averages/totals (only past days, no peeking ahead)
    for col in ["apparent_temperature_min", "apparent_temperature_max"]:
        t[col + "_yesterday"] = t[col].shift(1)
        t[col + "_3day"] = t[col].rolling(3).mean()
    t["feels_min_change"] = t["apparent_temperature_min"] - t["apparent_temperature_min"].shift(1)
    t["rain_3day"] = t["precipitation_sum"].rolling(3).sum()
    t["gust_yesterday"] = t["wind_gusts_10m_max"].shift(1)
    return t

df = add_features(df)
WEATHER_FEATURES = ["apparent_temperature_max", "apparent_temperature_min", "precipitation_sum",
                    "snowfall_sum", "wind_gusts_10m_max", "wind_speed_10m_max", "wind_dir_sin", "wind_dir_cos",
                    "weather_code", "month", "season_sin", "season_cos",
                    "apparent_temperature_min_yesterday", "apparent_temperature_min_3day",
                    "apparent_temperature_max_yesterday", "apparent_temperature_max_3day",
                    "feels_min_change", "rain_3day", "gust_yesterday"]
AIR_FEATURES = ["pm2_5", "ozone_8h", "pm10", "nitrogen_dioxide", "us_aqi"]
features = WEATHER_FEATURES + AIR_FEATURES

# Tomorrow's answers, moved up one row
target_cols = []
for h in hazards:
    df[h + "_tomorrow"] = df[h].shift(-1)
    target_cols.append(h + "_tomorrow")

df["tomorrow"] = df["time"] + pd.Timedelta(days=1)
tomorrow_is_school = is_school_day(df["tomorrow"])

W_TARGETS = [h + "_tomorrow" for h in WEATHER_HAZARDS]
A_TARGETS = [h + "_tomorrow" for h in AIR_HAZARDS]

# Keep every school day with weather; air columns may be empty before August 2022
data = df[tomorrow_is_school].dropna(subset=WEATHER_FEATURES + W_TARGETS).copy()
data[W_TARGETS] = data[W_TARGETS].astype(bool)
data["unsafe_tomorrow"] = data[target_cols].fillna(False).astype(bool).any(axis=1)
print("Rows we can learn from (weather):", len(data))
print("Rows that also have air data:", int(data.dropna(subset=AIR_FEATURES + A_TARGETS).shape[0]))

## Step 4: Split by time (never shuffle)

- **Train** on every school day **before** September 2025.
- **Test** on the **2025-26 school year**, which the model never sees while learning.

This is like a real forecast: you only ever know the past.

In [ ]:
TEST_START = "2025-09-01"
train = data[data["tomorrow"] < TEST_START]
test = data[data["tomorrow"] >= TEST_START]

print("Train:", train["tomorrow"].min().date(), "to", train["tomorrow"].max().date(), "-", len(train), "days")
print("Test: ", test["tomorrow"].min().date(), "to", test["tomorrow"].max().date(), "-", len(test), "days")
print("Unsafe share in train:", round(train["unsafe_tomorrow"].mean(), 2))
print("Unsafe share in test: ", round(test["unsafe_tomorrow"].mean(), 2))

## Step 4b: What brings pollution to Philadelphia?

Before predicting, we look at **what pollution goes along with** in the training years only (the test year stays untouched):

1. **Pollution rose:** the average AQI and PM2.5 for each **wind direction**. If wind from one direction brings dirtier air, that direction's bar is taller.
2. **Correlation:** a number from -1 to 1 for how AQI moves with each weather input. Near 1 means "goes up together," near -1 means "one goes up, the other goes down," and near 0 means "no clear link."

**Correlation is not cause.** Hot days and high ozone go together because sunlight makes ozone, but a number alone can't prove that. We use the results to explain, not to claim causes we didn't test.

AQI categories come from the US EPA: 0-50 Good, 51-100 Moderate, 101-150 Unhealthy for sensitive groups (which includes children), 151-200 Unhealthy, 201-300 Very unhealthy, 301+ Hazardous.

In [ ]:
import matplotlib.pyplot as plt

COMPASS = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
def compass(deg):
    return COMPASS[int(((deg % 360) + 22.5) // 45) % 8]

past = df[df["time"] < TEST_START].dropna(subset=["wind_direction_10m_dominant", "us_aqi"]).copy()
past["wind_from"] = past["wind_direction_10m_dominant"].map(compass)
rose = past.groupby("wind_from")[["us_aqi", "pm2_5"]].mean().reindex(COMPASS).round(1)
rose["days"] = past["wind_from"].value_counts().reindex(COMPASS)
print("Average pollution by wind direction (training years):")
print(rose)

links = past[["us_aqi", "apparent_temperature_max", "wind_speed_10m_max", "precipitation_sum", "month"]].corr()["us_aqi"]
print()
print("How AQI moves with each weather input (correlation, -1 to 1):")
print(links.drop("us_aqi").round(2).to_string())
if "main_pollutant" in past:
    print()
    print("Main pollutant, share of days:")
    print(past["main_pollutant"].value_counts(normalize=True).round(2).to_string())

plt.figure(figsize=(7, 3.5))
plt.bar(rose.index, rose["us_aqi"], color="#0F2A43")
plt.ylabel("Average daily US AQI")
plt.xlabel("Wind comes from")
plt.title("Philadelphia: average air quality by wind direction (2022-25)")
plt.savefig("pollution_by_wind.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 5: The baseline, "tomorrow will be the same as today"

The simplest possible guess: whatever danger happened today, guess it happens again tomorrow. Our model is only useful if it beats this.

In [ ]:
baseline_pred = test[hazards].fillna(False).values.astype(bool)   # today's dangers = guess for tomorrow

## Step 6: Train the model (two random forests)

A **random forest** is a group of decision trees. Each tree asks yes/no questions like "is today's 'feels like' max above 30 °C?" and the trees vote.

We train **two forests**, because each danger should learn from all the history that exists for it:
- **Weather forest**: too hot, too cold, heavy rain, dangerous wind, heavy snow. Learns from **30 years** (1995-2025) of weather inputs.
- **Air forest**: smoke/particles and ozone smog. Learns from the years with air data (2022-25), using weather **and** air inputs.

`RecessModel` keeps the two together, and `danger_probs()` asks both and returns one chance (0 to 1) per danger.

- `class_weight="balanced"` tells each forest that rare unsafe days matter, so it doesn't just say "safe" every time.
- `random_state=42` makes the result the same every time we run it. `n_jobs=-1` uses all the computer's cores, so training is faster.

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier

class RecessModel:
    # Holds our two forests: one for weather dangers, one for air dangers
    def __init__(self, weather_forest, air_forest):
        self.weather, self.air = weather_forest, air_forest

def make_forest():
    return RandomForestClassifier(n_estimators=200, min_samples_leaf=3,
                                  class_weight="balanced", random_state=42, n_jobs=-1)

def train_models(rows):
    weather_forest = make_forest().fit(rows[WEATHER_FEATURES], rows[W_TARGETS].astype(bool))
    air_rows = rows.dropna(subset=AIR_FEATURES + A_TARGETS)          # only days with air data
    air_forest = make_forest().fit(air_rows[features], air_rows[A_TARGETS].astype(bool))
    return RecessModel(weather_forest, air_forest)

def forest_probs(forest, X):
    # Chance (0 to 1) of "True" for each danger this forest predicts
    cols = []
    for p, classes in zip(forest.predict_proba(X), forest.classes_):
        classes = list(classes)
        cols.append(p[:, classes.index(True)] if True in classes else np.zeros(len(X)))
    return np.column_stack(cols)

def danger_probs(m, X):
    # One column per danger, in the same order as `hazards`
    return np.column_stack([forest_probs(m.weather, X[WEATHER_FEATURES]), forest_probs(m.air, X[features])])

model = train_models(train)
model_pred = danger_probs(model, test[features]) >= 0.5   # first try: warn if the chance is 50% or more

## Step 7: Model vs. baseline

For each danger we count, on the test year:
- **caught**: dangerous days we warned about (higher is better)
- **missed**: dangerous days we did NOT warn about (the worst mistake for kids)
- **false alarms**: we warned but it was fine (annoying, less harmful)

⚠️ If accuracy is above 95%, be suspicious and look for leakage.

In [ ]:
y_true = test[target_cols].fillna(False).values.astype(bool)

def score(true, pred):
    return {"caught": int((true & pred).sum()),
            "missed": int((true & ~pred).sum()),
            "false alarms": int((~true & pred).sum())}

rows = []
for i, h in enumerate(hazards):
    b = score(y_true[:, i], baseline_pred[:, i])
    m = score(y_true[:, i], model_pred[:, i])
    rows.append({"danger": h, "real days": int(y_true[:, i].sum()),
                 "baseline caught": b["caught"], "baseline missed": b["missed"], "baseline false alarms": b["false alarms"],
                 "model caught": m["caught"], "model missed": m["missed"], "model false alarms": m["false alarms"]})
print(pd.DataFrame(rows).to_string(index=False))

# Overall: is tomorrow unsafe for ANY reason?
true_any = y_true.any(axis=1)
base_any = baseline_pred.any(axis=1)
model_any = model_pred.any(axis=1)
print()
for name, pred in [("Baseline", base_any), ("Model", model_any)]:
    s = score(true_any, pred)
    acc = (pred == true_any).mean()
    print(f"{name}: accuracy {acc:.2f} | caught {s['caught']} | missed {s['missed']} | false alarms {s['false alarms']}")
    if acc > 0.95:
        print("   WARNING: above 95%. Check for leakage before trusting this!")

## Step 7b: Too many false alarms, so make the model less jumpy

**What we saw on our first runs (with only 3 years of history):** the first model missed few unsafe days but raised far too many false alarms. After tuning, it caught only 18 unsafe days vs. 47 for "tomorrow = today", and missed 42 of 45 unsafe winter days, because winter 2025-26 was much colder than anything in its 3 training years. That's why we moved to 30 years of weather history.

**The fix:** for each danger, the model gives a **chance from 0 to 1**. Until now it warned whenever the chance was above 0.5. We'll pick a better **cutoff**:

1. Hold back the **2024-25 school year** from the training data as a practice year ("validation").
2. Train a model on the years before it, and try cutoffs from 0.05 to 0.95.
3. Keep the **lowest cutoff whose false alarms are no more than the baseline's** on that year. Lowest means it catches as many unsafe days as possible.
4. Use that cutoff on the real test year (2025-26) **once**.

**Why not pick the cutoff using the test year?** That would be cheating: we'd be tuning on the answers. The test year stays untouched until the end.

**What happened with one shared cutoff (our 30-year run):** at a 0.65 cutoff the model caught 55 of 70 unsafe days but raised 142 false alarms (accuracy 0.42). The cause was rare dangers: at 0.5, heavy rain alone made 186 false alarms on only 3 real days, because `class_weight="balanced"` inflates rare dangers' chances. One cutoff can't fit both common dangers like cold and rare ones like heavy rain.

**So each danger now gets its own cutoff**, chosen on the **practice year only**. Our team's rule: **a missed dangerous day counts twice as bad as a false alarm** (`MISS_COST = 2`). For each danger we try cutoffs from 0.05 to 0.95 and keep the one with the lowest total "cost" (2 × missed + false alarms) on the practice year. If a danger never happened in the practice year, its cutoff stays very cautious (high).

In [ ]:
VAL_START = "2024-09-01"
fit_part = train[train["tomorrow"] < VAL_START]
val_part = train[train["tomorrow"] >= VAL_START]

tuner = train_models(fit_part)            # same recipe, but without the practice year

# Each danger gets its own cutoff, chosen on the PRACTICE year only.
# Team rule: a missed dangerous day counts as twice as bad as a false alarm.
MISS_COST = 2
val_probs = danger_probs(tuner, val_part[features])
val_truth = val_part[target_cols].fillna(False).values.astype(bool)
val_base_each = val_part[hazards].fillna(False).values.astype(bool)

CUTOFFS, rows = {}, []
for i, h in enumerate(hazards):
    best_cut, best_cost = 0.95, None
    for t in np.arange(0.05, 1.0, 0.05):
        s_ = score(val_truth[:, i], val_probs[:, i] >= t)
        cost = MISS_COST * s_["missed"] + s_["false alarms"]
        if best_cost is None or cost <= best_cost:    # on a tie, keep the higher (more cautious) cutoff
            best_cut, best_cost = round(float(t), 2), cost
    CUTOFFS[h] = best_cut
    s_ = score(val_truth[:, i], val_probs[:, i] >= best_cut)
    b_ = score(val_truth[:, i], val_base_each[:, i])
    rows.append({"danger": h, "cutoff": best_cut, "practice real days": int(val_truth[:, i].sum()),
                 "model caught": s_["caught"], "model false alarms": s_["false alarms"],
                 "baseline caught": b_["caught"], "baseline false alarms": b_["false alarms"]})
print("Cutoff for each danger, chosen on the practice year (2024-25):")
print(pd.DataFrame(rows).to_string(index=False))
CUT = np.array([CUTOFFS[h] for h in hazards])   # same order as `hazards`, so it lines up with danger_probs columns

# Use the cutoffs on the real test year, with the model trained on all training years
test_probs = danger_probs(model, test[features])
model_pred = test_probs >= CUT
model_any = model_pred.any(axis=1)
print()
for name, pred in [("Baseline", base_any), ("Model (cutoff per danger)", model_any)]:
    s = score(true_any, pred)
    acc = (pred == true_any).mean()
    print(f"{name}: accuracy {acc:.2f} | caught {s['caught']} | missed {s['missed']} | false alarms {s['false alarms']}")
    if acc > 0.95:
        print("   WARNING: above 95%. Check for leakage before trusting this!")

From here on, every step uses the **tuned** model.

## Step 8: Confusion matrix (overall "unsafe tomorrow")

A 2x2 table: rows are what really happened, columns are what the model said.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(true_any, model_any, display_labels=["Safe", "Unsafe"], cmap="Blues")
plt.title("Recess Ready: model on the 2025-26 school year")
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 9: Where does it fail?

Real unsafe days the model **missed**. We look at a few and ask why.

In [ ]:
missed = test[true_any & ~model_any]
print("Missed unsafe days:", len(missed))
for _, row in missed.head(3).iterrows():
    real = [h for h in hazards if row[h + "_tomorrow"]]
    print(f"- {row['tomorrow'].date()}: really {real}. "
          f"Today's feels-like max {row['apparent_temperature_max']:.0f} C, min {row['apparent_temperature_min']:.0f} C, "
          f"PM2.5 {row['pm2_5']:.0f}, gusts {row['wind_gusts_10m_max']:.0f} km/h")

## Step 10: Does it work equally well in every season?

Our fairness check: the same accuracy and missed-day counts, split by season.

In [ ]:
def season(month):
    if month in [9, 10, 11]: return "Fall"
    if month in [12, 1, 2]: return "Winter"
    if month in [3, 4, 5]: return "Spring"
    return "June"

seasons = test["tomorrow"].dt.month.map(season).values
rows = []
for s in ["Fall", "Winter", "Spring", "June"]:
    keep = seasons == s
    t, m, b = true_any[keep], model_any[keep], base_any[keep]
    rows.append({"season": s, "days": int(keep.sum()), "unsafe days": int(t.sum()),
                 "model accuracy": round((m == t).mean(), 2),
                 "baseline accuracy": round((b == t).mean(), 2),
                 "model missed": int((t & ~m).sum())})
print(pd.DataFrame(rows).to_string(index=False))

## Step 9b: Check against a real weather station (ground truth)

Open-Meteo's weather history is calculated by a computer weather model. How close is it to what **real instruments** measured? We download the **Philadelphia International Airport (PHL) weather station** for the test year from the Iowa Environmental Mesonet (Iowa State University; free, no key). These are the airport's real hourly reports (called METARs).

From the station we build the same dangers (too hot, too cold, heavy rain, dangerous wind) and compare:
1. How often do Open-Meteo and the real station **agree** about each danger?
2. How does our model score when the **real station** is the answer key?
3. How many **thunderstorm** and **freezing rain / ice pellet** days did the station report? Our model can't predict these.

Station units are US units (°F, inches, knots), so we convert them. Heavy snow isn't compared, because the station reports when it snows, not how much.

In [ ]:
import io

ASOS_URL = "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py"

def station_table():
    start, end = pd.Timestamp(TEST_START) - pd.Timedelta(days=5), pd.Timestamp(END) + pd.Timedelta(days=1)
    params = {"station": "PHL", "data": ["feel", "p01i", "gust", "wxcodes"],
              "year1": start.year, "month1": start.month, "day1": start.day,
              "year2": end.year, "month2": end.month, "day2": end.day,
              "tz": "America/New_York", "format": "onlycomma", "latlon": "no",
              "missing": "empty", "trace": "empty", "report_type": "3"}   # 3 = regular hourly reports
    for attempt in range(4):
        try:
            r = requests.get(ASOS_URL, params=params, timeout=180)
            r.raise_for_status()
            break
        except Exception as err:
            if attempt == 3:
                raise
            print("Station download problem, trying again...", type(err).__name__)
            time.sleep(10 * (attempt + 1))
    obs = pd.read_csv(io.StringIO(r.text))
    obs["time"] = pd.to_datetime(obs["valid"]).dt.floor("D")
    for col in ["feel", "p01i", "gust"]:
        obs[col] = pd.to_numeric(obs[col], errors="coerce")
    codes = obs["wxcodes"].fillna("")
    obs["thunder"] = codes.str.contains("TS")
    obs["frozen"] = codes.str.contains("FZRA|FZDZ|PL")
    day = obs.groupby("time").agg(feel_max=("feel", "max"), feel_min=("feel", "min"), rain_in=("p01i", "sum"),
                                  gust_kt=("gust", "max"), thunder=("thunder", "any"), frozen=("frozen", "any"))
    return day.reset_index()

station = cached("phl_station_daily", station_table)
station["too_hot"] = (station["feel_max"] - 32) * 5 / 9 >= 32       # °F to °C
station["too_cold"] = (station["feel_min"] - 32) * 5 / 9 <= -7
station["heavy_rain"] = station["rain_in"] * 25.4 >= 25              # inches to mm
station["high_wind"] = station["gust_kt"] * 1.852 >= 74              # knots to km/h

SHARED = ["too_hot", "too_cold", "heavy_rain", "high_wind"]
check = test[["tomorrow"] + [h + "_tomorrow" for h in SHARED]].merge(
    station.rename(columns={"time": "tomorrow"}), on="tomorrow", how="inner")
print("Test school days with station reports:", len(check))
print()
print("How often Open-Meteo and the real station agree:")
for h in SHARED:
    agree = (check[h + "_tomorrow"].astype(bool) == check[h].astype(bool)).mean()
    print(f"  {h}: {agree:.0%} of days   (station: {int(check[h].sum())} days, Open-Meteo: {int(check[h + '_tomorrow'].sum())} days)")

# Our model's weather warnings vs. the real station's answer
w_cols = [hazards.index(h) for h in SHARED]
model_w = pd.Series((danger_probs(model, test[features])[:, w_cols] >= CUT[w_cols]).any(axis=1), index=test["tomorrow"])
base_w = pd.Series(test[SHARED].values.astype(bool).any(axis=1), index=test["tomorrow"])
station_truth = check.set_index("tomorrow")[SHARED].astype(bool).any(axis=1)
print()
print("Scored against the REAL station (heat, cold, heavy rain, wind):")
for name, pred in [("Baseline", base_w), ("Model", model_w)]:
    p_ = pred.reindex(station_truth.index).values
    s_ = score(station_truth.values, p_)
    print(f"  {name}: accuracy {(p_ == station_truth.values).mean():.2f} | caught {s_['caught']} | "
          f"missed {s_['missed']} | false alarms {s_['false alarms']}")

print()
print("Days the station reported that our model can't predict:")
print("  Thunderstorm days:", int(check["thunder"].sum()))
print("  Freezing rain / ice pellet days:", int(check["frozen"].sum()))

## Step 10b: Does a Philadelphia model work across the whole region?

We trained only on Center City. Does it still work 20-60 km away, in New Jersey, Delaware and the Pennsylvania suburbs? This is our "does it hold up in a new place?" test, and it powers the regional danger chart in the app.

1. Make a **grid of 35 places** (every 0.2° of latitude and longitude) around Philadelphia.
2. Download the test year for all 35 places **in one request**, then reuse `daily_table`, `add_hazards` and `add_features`, so every place follows the exact same rules.
3. Run the **same model and cutoff** (no retraining) and score each place.

Weak spots on this list are real failures to report, not to hide.

In [ ]:
REG_LATS = [39.6, 39.8, 40.0, 40.2, 40.4]
REG_LONS = [-75.8, -75.6, -75.4, -75.2, -75.0, -74.8, -74.6]
points = [(la, lo) for la in REG_LATS for lo in REG_LONS]

REG_START = (pd.Timestamp(TEST_START) - pd.Timedelta(days=10)).strftime("%Y-%m-%d")
def region_table():
    reg_weather, reg_air = download([p[0] for p in points], [p[1] for p in points], REG_START, END)
    parts = []
    for (la, lo), w, a in zip(points, reg_weather, reg_air):
        part = daily_table(w, a)
        part["lat"], part["lon"] = la, lo
        parts.append(part)
    return pd.concat(parts)

region_raw = cached("region_daily", region_table)

reg_tables = []
for (la, lo), part in region_raw.groupby(["lat", "lon"]):
    t = add_features(add_hazards(part.sort_values("time").reset_index(drop=True)))
    t["tomorrow"] = t["time"] + pd.Timedelta(days=1)
    for h in hazards:
        t[h + "_tomorrow"] = t[h].shift(-1)
    reg_tables.append(t)
region = pd.concat(reg_tables)
region = region[is_school_day(region["tomorrow"]) & (region["tomorrow"] >= TEST_START)]
region = region.dropna(subset=features + target_cols).copy()
region[target_cols] = region[target_cols].astype(bool)

# One prediction for every place and day at once (much faster than one at a time)
reg_probs = danger_probs(model, region[features])
region["warn"] = (reg_probs >= CUT).any(axis=1)
region["really_unsafe"] = region[target_cols].any(axis=1)

rows = []
for (la, lo), g in region.groupby(["lat", "lon"]):
    rows.append({"lat": la, "lon": lo, "accuracy": round((g["warn"] == g["really_unsafe"]).mean(), 2),
                 "unsafe days": int(g["really_unsafe"].sum()),
                 "missed": int((g["really_unsafe"] & ~g["warn"]).sum())})
place_scores = pd.DataFrame(rows)
print("Accuracy across the 35 places: from", place_scores["accuracy"].min(), "to", place_scores["accuracy"].max())
print("Philadelphia (Center City) accuracy, for comparison:", round((model_any == true_any).mean(), 2))
print()
print("5 weakest places:")
print(place_scores.sort_values("accuracy").head(5).to_string(index=False))

## Step 11: The demo app

Pick a school day from the test year. The app shows what Recess Ready would have told the school **the evening before**, and what really happened.

The app has four pieces, each made by one small function that returns a bit of web page (HTML) as text:
- `meters_html()`: a bar for each danger showing the model's **chance** (0-100%), with our cutoff marked. A bar past the line means a warning.
- `map_html()`: a Life360-style map of Philadelphia. `stitch()` downloads small map pictures ("tiles") from OpenStreetMap **once** and glues them into one picture. `to_pixel()` uses standard map math (Web Mercator) to turn latitude/longitude into a spot on that picture, so we can place the school pin, the wind pin and the dashed path. Open-Meteo gives one reading for the whole city, so we show one point, not a fake reading for each neighborhood.
- `week_html()`: two small charts of the **7 days before the call**, feels-like temperature and PM2.5, against their danger lines.
- `chart_html()`: a **regional danger chart** in the style of the National Weather Service forecast chart. Each square is one of the 35 places from Step 10b, shaded by its most likely danger if that chance passes our cutoff (stripes = NWS-style "possible" areas), with a wind arrow at each place. Unlike the NWS chart, we don't draw fronts or high/low pressure: our data can't support that honestly.
- `air_html()`: today's **US AQI** in the official EPA colors, what the level means for kids (EPA wording, simplified), the **main pollutant**, and today's PM2.5, PM10, ozone and NO₂.
- `rose_html()`: the **pollution rose** from Step 4b, with tomorrow's wind direction highlighted.
- `recess_check()` puts it together for the chosen date.

*Prototype, not official guidance. Always follow official National Weather Service alerts and your school district.*

In [ ]:
import gradio as gr
import html

NAMES = {"too_hot": ("🔥", "Too hot"), "too_cold": ("🥶", "Too cold"),
         "bad_air": ("😷", "Smoke / particles"), "ozone_smog": ("🌫️", "Ozone smog"),
         "heavy_rain": ("🌊", "Flash-flood rain"), "high_wind": ("🌀", "Dangerous wind"),
         "thunderstorm": ("⚡", "Lightning"), "heavy_snow": ("❄️", "Heavy snow"),
         "ice": ("🧊", "Ice")}
COMPASS = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]

# Small line icons we drew ourselves (24 x 24 grid, same stroke for all), used instead of emoji
CLOUD = '<path d="M7 15a4 4 0 0 1-.4-7.98 5.5 5.5 0 0 1 10.7 1.3A3.4 3.4 0 0 1 17 15z"/>'
ICON_PATHS = {
    "too_hot": '<path d="M10 4a2 2 0 0 1 4 0v9.3a4 4 0 1 1-4 0z"/><path d="M12 13v3"/><path d="M18 4.5l1.5-1.5M19 9h2"/>',
    "too_cold": '<path d="M12 3v18M4.2 7.5l15.6 9M19.8 7.5l-15.6 9"/><path d="M9.5 4.5 12 7l2.5-2.5M9.5 19.5 12 17l2.5 2.5"/>',
    "bad_air": CLOUD + '<path d="M8 19h.01M12 20h.01M16 19h.01"/>',
    "ozone_smog": '<circle cx="12" cy="9" r="3.5"/><path d="M12 2.5V4M5.5 9H4M20 9h-1.5M7.4 4.4l1 1M16.6 4.4l-1 1"/><path d="M4 16h16M6 20h12"/>',
    "heavy_rain": CLOUD + '<path d="M8 18l-1 3M12 18l-1 3M16 18l-1 3"/>',
    "high_wind": '<path d="M3 8h10a3 3 0 1 0-3-3"/><path d="M3 12h15a3 3 0 1 1-3 3"/><path d="M3 16h7"/>',
    "heavy_snow": CLOUD + '<path d="M8 18.5h.01M12 20.5h.01M16 18.5h.01M10 21.5h.01M14 21.5h.01"/>',
    "wind": '<path d="M3 8h10a3 3 0 1 0-3-3"/><path d="M3 12h15a3 3 0 1 1-3 3"/><path d="M3 16h7"/>',
    "arrow": '<path d="M4 12h15M13 6l6 6-6 6"/>',
}
def icon(name, size=18):
    return (f'<svg class="ico" viewBox="0 0 24 24" width="{size}" height="{size}" fill="none" stroke="currentColor" '
            f'stroke-width="2" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true">{ICON_PATHS[name]}</svg>')


# Real test-year scores (calculated from the results above, never typed in by hand)
m_score = score(true_any, model_any)
b_score = score(true_any, base_any)
total_unsafe = int(true_any.sum())
test_dates = test["tomorrow"].dt.strftime("%Y-%m-%d").tolist()


# ---------- Piece 1: danger meters (the model's chance for each danger) ----------
def meters_html(chances, happened):
    rows = ""
    order = sorted(range(len(hazards)), key=lambda i: -chances[i])   # most likely first
    for i in order:
        h = hazards[i]
        name = NAMES[h][1]
        pct = round(chances[i] * 100)
        hot = "hot" if chances[i] >= CUTOFFS[h] else ""
        tag = '<span class="tag">happened</span>' if h in happened else ""
        rows += (f'<div class="meter {hot}"><span class="mname">{icon(h)} {name} {tag}</span>'
                 f'<span class="track"><span class="fill" style="width:{pct}%"></span>'
                 f'<span class="cut" style="left:{CUTOFFS[h]*100:.0f}%" title="Cutoff {CUTOFFS[h]*100:.0f}%"></span></span>'
                 f'<span class="pct"><span data-count="{pct}">{pct}</span>%</span></div>')
    return (f'<div class="board"><p class="board-head">Chance of each danger tomorrow. '
            f'The white line marks each danger\'s own cutoff.</p>{rows}</div>')


# ---------- Piece 2: the close-up "Life360" map (live: zoom, drag, moving wind) ----------
# Python only prepares the data. project/static/recess_maps.js draws the map in the browser
# with Leaflet (a free map library we ship in the project) and OpenStreetMap map tiles.
import json, math

def winds_for(tomorrow):
    # Today's wind at each of the 35 places, as "east" (u) and "north" (v) speeds in km/h.
    # Wind direction is where the wind comes FROM, so it blows toward the opposite side.
    rows = region[region["tomorrow"] == tomorrow]
    out = []
    for _, r in rows.iterrows():
        d, speed = math.radians(r["wind_direction_10m_dominant"]), r["wind_speed_10m_max"]
        out.append({"lat": r["lat"], "lon": r["lon"], "speed": round(speed), "from": compass(r["wind_direction_10m_dominant"]),
                    "u": round(-speed * math.sin(d), 2), "v": round(-speed * math.cos(d), 2)})
    return out

def map_box(kind, config, css_class):
    config["kind"] = kind
    # The box is left empty on purpose: the script draws the map inside it
    return f'<div class="rrmap {css_class}" data-config="{html.escape(json.dumps(config))}"></div>'

def map_html(row, state):
    wind_from, gust = row["wind_direction_10m_dominant"], row["wind_gusts_10m_max"]
    up_lat = LAT + 0.09 * math.cos(math.radians(wind_from))
    up_lon = LON + 0.09 * math.sin(math.radians(wind_from)) / math.cos(math.radians(LAT))
    config = {"lat": LAT, "lon": LON, "up_lat": up_lat, "up_lon": up_lon, "winds": winds_for(row["tomorrow"])}
    pill = "Indoor recess" if state == "indoor" else "Outdoor recess"
    return f"""
    <div class="card mapcard"><p class="card-head">Where today's air is coming from</p>
      <div class="mapwrap">
        {map_box("close", config, "close")}
        <div class="chip">{icon("wind", 16)} Wind from the {compass(wind_from)}, gusts {gust:.0f} km/h</div>
      </div>
      <div class="sheet"><span class="handle"></span>
        <div class="sheet-row"><div><b>Center City, Philadelphia</b>
          <p>Today: PM2.5 {row["pm2_5"]:.0f} µg/m³, feels-like high {row["apparent_temperature_max"]:.0f}°C</p></div>
          <span class="pill {state}">{pill}</span></div></div>
      <p class="attr">Drag to move, scroll or use + / − to zoom. Arrows show today's wind: longer, faster and
      warmer-colored = stronger. Hover an arrow for details. Map © OpenStreetMap contributors.</p></div>"""


# ---------- Piece 3: small line charts of the week before ----------
def line_chart(values, lines, unit):
    # values: dict name -> (list of numbers, color); lines: list of (danger level, label)
    w, h, pad = 300, 120, 22
    allv = [v for vals, _ in values.values() for v in vals] + [lvl for lvl, _ in lines]
    lo, hi = min(allv), max(allv)
    span = (hi - lo) or 1
    x = lambda i: pad + i * (w - 2 * pad) / 6
    y = lambda v: h - pad - (v - lo) / span * (h - 2 * pad)
    svg = f'<svg viewBox="0 0 {w} {h}" class="chart" role="img">'
    for lvl, lab in lines:
        svg += (f'<line x1="{pad}" x2="{w-pad}" y1="{y(lvl):.1f}" y2="{y(lvl):.1f}" class="danger"/>'
                f'<text x="{w-pad}" y="{y(lvl)-4:.1f}" text-anchor="end" class="dlab">{lab}</text>')
    for name, (vals, color) in values.items():
        pts = " ".join(f"{x(i):.1f},{y(v):.1f}" for i, v in enumerate(vals))
        svg += f'<polyline points="{pts}" fill="none" stroke="{color}" stroke-width="2.5" pathLength="1" class="draw"/>'
        svg += f'<circle cx="{x(len(vals)-1):.1f}" cy="{y(vals[-1]):.1f}" r="3.5" fill="{color}"/>'
        svg += (f'<text x="{x(len(vals)-1)+6:.1f}" y="{y(vals[-1])+4:.1f}" class="vlab" fill="{color}">'
                f'{vals[-1]:.0f}{unit}</text>')
    svg += f'<text x="{pad}" y="{h-4}" class="axis">7 days ago</text><text x="{w-pad}" y="{h-4}" text-anchor="end" class="axis">today</text>'
    return svg + "</svg>"

def week_html(tomorrow):
    week = df[(df["time"] > tomorrow - pd.Timedelta(days=8)) & (df["time"] < tomorrow)].tail(7)
    temp = line_chart({"max": (week["apparent_temperature_max"].tolist(), "#C77C00"),
                       "min": (week["apparent_temperature_min"].tolist(), "#0F2A43")},
                      [(32, "too hot"), (-7, "too cold")], "°")
    air = line_chart({"pm": (week["pm2_5"].tolist(), "#C77C00")}, [(35.4, "unhealthy")], "")
    return (f'<div class="card"><p class="card-head">What Recess Ready saw this week</p>'
            f'<p class="chart-title">Feels-like temperature, °C (amber = daily high, navy = daily low)</p>{temp}'
            f'<p class="chart-title">PM2.5 particles, µg/m³</p>{air}</div>')


# ---------- Piece 5: today's air quality (official US EPA AQI scale and colors) ----------
AQI_LEVELS = [(50, "Good", "#00E400", "Air is fine for outdoor play."),
              (100, "Moderate", "#FFFF00", "Air is fine for most kids. A few very sensitive kids may notice it."),
              (150, "Unhealthy for sensitive groups", "#FF7E00",
               "Kids with asthma or other lung problems should limit long or hard outdoor play."),
              (200, "Unhealthy", "#FF0000", "Everyone should limit long or hard outdoor play."),
              (300, "Very unhealthy", "#8F3F97", "Everyone should avoid long or hard outdoor play."),
              (9999, "Hazardous", "#7E0023", "Everyone should stay indoors.")]
POLLUTANT_NAMES = {"pm2_5": "PM2.5 particles", "pm10": "PM10 particles", "ozone": "ozone",
                   "nitrogen_dioxide": "nitrogen dioxide", "sulphur_dioxide": "sulfur dioxide",
                   "carbon_monoxide": "carbon monoxide"}

def aqi_level(value):
    for top, name, color, advice in AQI_LEVELS:
        if value <= top:
            return name, color, advice

def air_html(row):
    aqi = row["us_aqi"]
    name, color, advice = aqi_level(aqi)
    main = POLLUTANT_NAMES.get(row.get("main_pollutant", ""), "not available")
    text_color = "#0F2A43" if name in ("Good", "Moderate", "Unhealthy for sensitive groups") else "#FFFFFF"
    # Draw the scale to size: each color band is as wide as its AQI range (the 200-300 band is twice as wide)
    bars, low = "", 0
    for top, lvl, col, _ in AQI_LEVELS[:5]:
        bars += f'<span style="background:{col};flex:{top - low}" title="{lvl}"></span>'
        low = top
    marker = min(aqi, 300) / 300 * 100
    ticks = "".join(f'<span style="left:{v / 300 * 100:.1f}%">{v}</span>' for v in [0, 50, 100, 150, 200, 300])
    return f"""
    <div class="card"><p class="card-head">Air quality today (US AQI)</p>
      <div class="aqi-top"><span class="aqi-num{" alert" if aqi > 100 else ""}" style="background:{color};color:{text_color}"><span data-count="{aqi:.0f}">{aqi:.0f}</span></span>
        <div><b class="aqi-name">{name}</b><p class="aqi-advice">{advice}</p></div></div>
      <div class="aqi-scale">{bars}<i style="left:{marker:.0f}%"></i></div>
      <div class="aqi-scale-l">{ticks}</div>
      <p class="aqi-main">Main pollutant today: <b>{main}</b></p>
      <ul class="pollutants">
        <li><span>PM2.5</span><b data-count="{row["pm2_5"]:.0f}">{row["pm2_5"]:.0f}</b> µg/m³</li><li><span>PM10</span><b data-count="{row["pm10"]:.0f}">{row["pm10"]:.0f}</b> µg/m³</li>
        <li><span>Ozone</span><b data-count="{row["ozone_8h"]:.0f}">{row["ozone_8h"]:.0f}</b> µg/m³</li><li><span>NO₂</span><b data-count="{row["nitrogen_dioxide"]:.0f}">{row["nitrogen_dioxide"]:.0f}</b> µg/m³</li>
      </ul></div>"""

# ---------- Piece 6: pollution rose (average AQI by wind direction, training years) ----------
def rose_html(wind_from):
    today = compass(wind_from)
    biggest = rose["us_aqi"].max()
    petals, labels = "", ""
    for i, d in enumerate(COMPASS):
        v = rose.loc[d, "us_aqi"]
        r = 22 + v / biggest * 88                     # petal length grows with average AQI
        a0, a1 = math.radians(i * 45 - 19), math.radians(i * 45 + 19)
        pts = f"0,0 {r * math.sin(a0):.1f},{-r * math.cos(a0):.1f} {r * math.sin(a1):.1f},{-r * math.cos(a1):.1f}"
        cls = "petal now" if d == today else "petal"
        petals += (f'<path class="{cls}" style="animation-delay:{i * 60}ms" '
                   f'd="M{pts.split()[0]} L{pts.split()[1]} A{r:.1f},{r:.1f} 0 0 1 {pts.split()[2]} Z">'
                   f'<title>Wind from the {d}: average AQI {v:.0f}</title></path>')
        lx, ly = 128 * math.sin(math.radians(i * 45)), -128 * math.cos(math.radians(i * 45))
        labels += f'<text x="{lx:.0f}" y="{ly + 4:.0f}" class="rlab{" now" if d == today else ""}">{d} {v:.0f}</text>'
    rings = "".join(f'<circle r="{22 + f * 88:.0f}" class="ring"/>' for f in (0.5, 1.0))
    return f"""
    <div class="card"><p class="card-head">What brings pollution to Philadelphia?</p>
      <p class="chart-title">Average AQI by the direction the wind comes from (2022-25). Longer petal = dirtier air.
      Today's wind: <b>from the {today}</b>.</p>
      <svg viewBox="-165 -150 330 300" class="rose" role="img" aria-label="Pollution rose">{rings}{petals}{labels}</svg>
      <p class="chart-title">Correlation shows what goes together, not what causes it.</p></div>"""


# ---------- Piece 4: regional danger chart (in the style of the NWS forecast chart) ----------
# NWS-like colors. True = striped (like "Possible" areas on the NWS chart)
STYLE = {"too_hot": ("#E4572E", True), "too_cold": ("#5DA9E9", False), "bad_air": ("#7B5EA7", False),
         "ozone_smog": ("#B08FD8", True), "heavy_rain": ("#D7263D", True), "high_wind": ("#B8860B", False),
         "thunderstorm": ("#E6C700", True), "heavy_snow": ("#8A99A8", True), "ice": ("#E26BB5", True)}

for i, h in enumerate(hazards):                 # keep each danger's chance next to its row
    region[h + "_chance"] = reg_probs[:, i]

LIKELY = 0.85   # solid area = "likely"; striped area = "possible" (at or above our cutoff)

def danger_areas(rows):
    # Turn the 35 places' chances into smooth outlined areas, like contour lines on a weather map:
    # 1) put each danger's chances on a 5 x 7 grid, 2) smooth it into a fine grid,
    # 3) trace the outline where the chance crosses our cutoff (and the "likely" level).
    from scipy.ndimage import zoom, gaussian_filter
    import matplotlib.pyplot as plt
    lats, lons = sorted(REG_LATS), sorted(REG_LONS)
    # One extra ring of "0% chance" around the grid, so areas round off instead of ending in a flat edge
    pad_lats = [lats[0] - 0.2] + lats + [lats[-1] + 0.2]
    pad_lons = [lons[0] - 0.2] + lons + [lons[-1] + 0.2]
    fine_lat = np.linspace(pad_lats[0], pad_lats[-1], (len(pad_lats) - 1) * 10 + 1)
    fine_lon = np.linspace(pad_lons[0], pad_lons[-1], (len(pad_lons) - 1) * 10 + 1)
    areas = []
    for h in hazards:
        grid = np.zeros((len(lats), len(lons)))
        for _, r in rows.iterrows():
            grid[lats.index(r["lat"]), lons.index(r["lon"])] = r[h + "_chance"]
        if grid.max() < CUTOFFS[h]:
            continue                                   # this danger isn't expected anywhere today
        grid = np.pad(grid, 1)                         # add the ring of zeros
        fine = zoom(grid, (len(fine_lat) / grid.shape[0], len(fine_lon) / grid.shape[1]), order=1)
        fine = gaussian_filter(fine, sigma=3)
        fine = np.clip(fine * grid.max() / max(fine.max(), 1e-9), 0, 1)   # keep the peak's real height
        for level, cut in [("possible", CUTOFFS[h]), ("likely", max(LIKELY, CUTOFFS[h]))]:
            if fine.max() < cut:
                continue
            fig, ax = plt.subplots()
            cs = ax.contourf(fine_lon, fine_lat, fine, levels=[cut, 1.01])    # everything at or above `cut`
            plt.close(fig)
            rings = [[[round(lat, 4), round(lon, 4)] for lon, lat in ring]
                     for path in cs.get_paths() for ring in path.to_polygons() if len(ring) > 2]
            if rings:
                areas.append({"name": h, "label": NAMES[h][1], "level": level, "color": STYLE[h][0],
                              "rings": rings, "pct": round(grid.max() * 100)})
    return areas

def chart_html(tomorrow):
    rows = region[region["tomorrow"] == tomorrow]
    areas = danger_areas(rows) if len(rows) else []
    seen = list(dict.fromkeys(a["name"] for a in areas))
    config = {"lat": LAT, "lon": LON, "areas": areas, "winds": winds_for(tomorrow),
              "stripes": {h: c for h, (c, striped) in STYLE.items()},
              "bounds": [[min(REG_LATS) - 0.12, min(REG_LONS) - 0.12], [max(REG_LATS) + 0.12, max(REG_LONS) + 0.12]]}
    legend = "".join(f'<li><span class="sw" style="background:{STYLE[h][0]}"></span>{NAMES[h][1]}</li>' for h in seen)
    legend = legend or '<li>No dangers expected</li>'
    day = pd.Timestamp(tomorrow).strftime("%A, %B %-d, %Y")
    summary = ("No dangers expected anywhere in the region." if not seen else
               "Dangers expected: " + ", ".join(NAMES[h][1].lower() for h in seen) + ".")
    return f"""
    <section class="chartsec"><h3>Regional danger chart</h3>
      <p class="chartsub">Our Philadelphia model, run on 35 places across the region for {day}. {summary}
      Hover over an area to see its chance.</p>
      <div class="nws">
        <div class="nwstitle">Recess Ready danger chart<br><span>Valid for school on {day}</span></div>
        {map_box("region", config, "region")}
        <ul class="legend">{legend}
          <li class="lg-note"><span class="sw hatch"></span>Striped: possible (past that danger's cutoff)</li>
          <li class="lg-note"><span class="sw solid"></span>Solid: likely ({LIKELY * 100:.0f}%+ chance)</li>
          <li class="lg-note"><span class="arrow-key">{icon("arrow", 16)}</span>Today's wind: longer and warmer = stronger</li></ul>
      </div>
      <p class="attr">Areas are smoothed between our 35 places, so edges are estimates. Map © OpenStreetMap contributors.
      Prototype, not official guidance; for official forecasts see weather.gov.</p>
    </section>"""


# ---------- Putting it together for one date ----------
def recess_check(date_str):
    idx = test.index[test["tomorrow"].dt.strftime("%Y-%m-%d") == date_str][0]
    row = test.loc[idx]
    chances = danger_probs(model, test.loc[[idx], features])[0]
    warned = [h for h, c in zip(hazards, chances) if c >= CUTOFFS[h]]
    happened = [h for h in hazards if row[h + "_tomorrow"]]
    day = row["tomorrow"].strftime("%A, %B %-d")

    if warned:
        state, call = "indoor", "Plan indoor recess"
        why = "Watch out for " + ", ".join(NAMES[h][1].lower() for h in warned) + "."
    else:
        state, call = "outdoor", "Outdoor recess is on"
        why = "No dangers expected tomorrow."
    if warned and happened and not set(warned) & set(happened):
        verdict = "Right that it was unsafe, but for the wrong reason."
    elif (len(warned) > 0) == (len(happened) > 0):
        verdict = "Recess Ready got this day right."
    elif happened:
        verdict = "Missed. Recess Ready said it was safe."
    else:
        verdict = "False alarm. The day was actually safe."
    truth = ", ".join(NAMES[h][1].lower() for h in happened) if happened else "a safe day"

    return f"""
    <div class="result">
      <div class="call-side">
        <h2 class="call {state}"><span>{call}</span></h2>
        <p class="why"><b>{day}.</b> {why}</p>
        <div class="truth"><p>What really happened: <b>{truth}</b></p><p class="verdict">{verdict}</p></div>
      </div>
      {meters_html(chances, happened)}
    </div>
    <div class="panels">
      {map_html(row, state)}
      {week_html(row["tomorrow"])}
    </div>
    <div class="panels">
      {air_html(row)}
      {rose_html(row["wind_direction_10m_dominant"])}
    </div>
    {chart_html(row["tomorrow"])}"""

HERO = """
<nav class="nav"><span class="logo"><span class="mark">R</span>Recess Ready</span>
  <span class="navlinks"><a href="#how">How it works</a><a href="#scores">Results</a></span></nav>
<section class="hero">
  <h1>The recess call,<br>made the <em>night before.</em></h1>
  <p class="lede">Heat, cold, smoke, smog, heavy rain, wind and snow. Recess Ready checks seven dangers for
  Philadelphia schools and tells principals tonight whether tomorrow's recess should move inside.</p>
</section>"""

def scores_html():
    def row(name, s, cls):
        return (f'<div class="srow {cls}"><span class="sname">{name}</span>'
                f'<span><b data-count="{s["caught"]}">{s["caught"]}</b> caught</span><span><b data-count="{s["missed"]}">{s["missed"]}</b> missed</span>'
                f'<span><b data-count="{s["false alarms"]}">{s["false alarms"]}</b> false alarms</span></div>')
    return (f'<section class="scores" id="scores"><h3>How it did on the 2025-26 school year</h3>'
            f'<p>{total_unsafe} school days were really unsafe. The model never saw this year while learning.</p>'
            + row("Recess Ready", m_score, "ours")
            + row("“Tomorrow = today” guess", b_score, "base")
            + f'<p class="regline">Across 35 places around Philadelphia, accuracy ranged from <b>{place_scores["accuracy"].min():.2f}</b> to <b>{place_scores["accuracy"].max():.2f}</b> (Center City: <b>{(model_any == true_any).mean():.2f}</b>).</p>'
            + '</section><section class="how" id="how"><h3>How it works</h3>'
            '<p>Every evening we take today’s weather, wind and air in Philadelphia from Open-Meteo, and a random forest '
            'predicts the chance of each danger tomorrow. If any chance passes a cutoff we chose on last school year, '
            'the call is indoor recess. The map shows our single data point for the city: Open-Meteo’s data is too '
            'coarse to give each neighborhood its own reading.</p></section>'
            '<p class="fine">Prototype, not official guidance. Always follow National Weather Service alerts and your '
            'school district. Air data is a computer-model estimate (CAMS via Open-Meteo), not a sensor reading. '
            'Map © OpenStreetMap contributors.</p>')

CSS = """
:root, .dark { --ease: cubic-bezier(.16, 1, .3, 1);
  --shadow-lg: 0 18px 40px -14px rgba(15,42,67,.35), 0 2px 6px rgba(15,42,67,.08);
  --shadow-xl: 0 26px 56px -16px rgba(15,42,67,.42), 0 3px 8px rgba(15,42,67,.1);
  --glass: rgba(255,255,255,.72);
  --cream:#EAF2F8; --ink:#0F2A43; --orange:#F5B700; --amber:#C77C00; --green:#2E8B57; --paper:#FFFFFF; --line:#D3E0EA; --muted:#4A6278;
  --body-background-fill: var(--cream); --background-fill-primary: var(--paper);
  --block-background-fill: var(--paper); --input-background-fill: #FFFFFF;
  --body-text-color: var(--ink); --block-label-text-color: var(--muted); --border-color-primary: var(--line);
  --input-border-color: var(--line); --block-border-color: var(--line); color-scheme: light; }
html, body, gradio-app, .gradio-container, .main, .wrap { background: var(--cream) !important; }
body, .gradio-container { background: var(--cream) !important; color: var(--ink) !important;
  font-family: 'Geist', system-ui, sans-serif !important; }
html { font-size: clamp(15px, 0.3vw + 13.5px, 18px); }
.gradio-container { max-width: min(1320px, 94vw) !important; margin-inline: auto !important; padding-inline: clamp(12px, 3vw, 32px) !important; }
footer { display: none !important; }
.nav { display:flex; justify-content:space-between; align-items:center; padding: 0.4rem 0 1.6rem; }
.logo { display:flex; gap:0.55rem; align-items:center; font-weight:700; font-size:1.05rem; color:var(--ink); }
.mark { background:var(--ink); color:var(--cream); width:28px; height:28px; border-radius:7px;
  display:grid; place-items:center; font-size:0.9rem; }
.navlinks a { color:var(--muted) !important; text-decoration:none; margin-left:1.6rem; font-size:0.95rem; }
.hero h1 { font-size: clamp(2.6rem, 6vw, 4.6rem); line-height:1.0; letter-spacing:-0.035em; font-weight:700;
  color:var(--ink) !important; margin:0 0 1rem; }
.hero h1 em { font-family:'Instrument Serif', Georgia, serif; font-weight:400; letter-spacing:-0.01em;
  text-decoration: underline; text-decoration-color: var(--orange); text-decoration-thickness: 3px; text-underline-offset: 0.12em; }
.hero .lede { font-size:1.15rem; line-height:1.55; color:var(--muted) !important; max-width:56ch; margin:0 0 1.4rem; }
#picker { background:transparent !important; border:none !important; gap:0.8rem; align-items:end; }
#picker .block { border-radius:14px !important; }
#picker label, #picker label span, #picker [data-testid='block-info'] { color:var(--muted) !important; }
#go { background:var(--orange) !important; color:var(--ink) !important; border:none !important; border-radius:999px !important;
  font-weight:600 !important; font-size:1.05rem !important; min-height:52px; }
#go:hover { background:#E0A600 !important; }
.result { display:grid; grid-template-columns: 1fr 1.15fr; gap:1.4rem; margin-top:0.6rem; }
.call-side, .card { background:var(--paper); border:1px solid var(--line); border-radius:20px; padding:1.6rem; }
.call { font-size: clamp(1.9rem, 3.2vw, 2.7rem); letter-spacing:-0.03em; line-height:1.05; margin:0.3rem 0 0.6rem; font-weight:700; }
.call.indoor { color:var(--ink) !important; } .call.indoor span { background: linear-gradient(transparent 58%, var(--orange) 58%);
  -webkit-box-decoration-break: clone; box-decoration-break: clone; }
.call.outdoor { color:var(--green) !important; }
.why { font-size:1.12rem; margin:0 0 1.4rem; color:var(--ink) !important; }
.truth { border-top:1px solid var(--line); padding-top:1rem; }
.truth p { margin:0.2rem 0; color:var(--muted) !important; } .truth b { color:var(--ink); }
.truth .verdict { color:var(--ink) !important; font-weight:600; }
.board { background:var(--ink); border-radius:20px; padding:1.4rem 1.5rem; box-shadow: var(--shadow-lg); }
.board-head { color:#A9BDD0 !important; margin:0 0 0.8rem; font-size:0.92rem; }
.meter { border-radius:10px; transition: background .2s ease, padding .2s ease; display:grid; grid-template-columns: minmax(8.5rem, 11.5rem) 1fr 3rem; gap:0.7rem; align-items:center; padding:0.32rem 0; }
.meter span { color:#B9CBDB !important; font-size:0.92rem; }
.meter.hot .mname, .meter.hot .pct { color:#fff !important; font-weight:600; }
.track { position:relative; height:10px; background:#1C3A57; border-radius:99px; }
.fill { position:absolute; left:0; top:0; bottom:0; background:#5C7A96; border-radius:99px; }
.meter.hot .fill { background:var(--orange); }
.cut { position:absolute; top:-4px; bottom:-4px; width:2px; background:#FFFFFF; }
.pct { text-align:right; font-variant-numeric: tabular-nums; }
.tag { font-size:0.7rem !important; border:1px solid currentColor; border-radius:99px; padding:0 0.45em; margin-left:0.3em; }
.panels { display:grid; grid-template-columns: 1.25fr 1fr; gap:1.4rem; margin-top:1.4rem; }
.card-head { font-weight:600; margin:0 0 0.8rem; color:var(--ink) !important; font-size:1.05rem; }
.mapcard { padding:1.2rem; }
.mapwrap { position:relative; }
.chip { position:absolute; z-index:1001; top:12px; left:12px; background:#fff; color:var(--ink) !important; border-radius:999px;
  padding:0.4rem 0.8rem; font-size:0.85rem; font-weight:600; box-shadow:0 4px 14px rgba(15,42,67,.18); }
.up-dot { display:grid; place-items:center; width:34px; height:34px; border-radius:50%; background:#F5B700;
  border:3px solid #fff; box-shadow:0 4px 12px rgba(15,42,67,.3); font-size:1rem; }
.avatar { position:relative; display:grid; place-items:center; width:52px; height:52px; border-radius:50%;
  background:var(--ink); border:4px solid #fff; box-shadow:0 6px 18px rgba(15,42,67,.35); font-size:1.5rem; z-index:2; }
.avatar::after { content:""; position:absolute; bottom:-11px; left:50%; transform:translateX(-50%);
  border:8px solid transparent; border-top:9px solid #fff; }
.pulse { position:absolute; left:50%; bottom:-14px; width:26px; height:26px; margin-left:-13px; border-radius:50%;
  background:rgba(15,42,67,.25); animation: pulse 2s ease-out infinite; }
@keyframes pulse { from { transform:scale(.6); opacity:.9; } to { transform:scale(3.2); opacity:0; } }
@media (prefers-reduced-motion: reduce) { .pulse { animation:none; opacity:.4; } }
.sheet { position:absolute; left:10px; right:10px; bottom:10px; background:#fff; border-radius:18px;
  padding:0.5rem 1rem 0.8rem; box-shadow:0 -2px 18px rgba(15,42,67,.18); }
.handle { display:block; width:38px; height:4px; border-radius:99px; background:#C9D6E2; margin:0 auto 0.5rem; }
.sheet-row { display:flex; justify-content:space-between; align-items:center; gap:0.8rem; }
.sheet b { color:var(--ink) !important; font-size:0.98rem; } .sheet p { margin:0.1rem 0 0; color:var(--muted) !important; font-size:0.85rem; }
.pill { flex:none; border-radius:999px; padding:0.35rem 0.75rem; font-size:0.82rem; font-weight:700; }
.pill.indoor { background:#F5B700; color:var(--ink) !important; } .pill.outdoor { background:#2E8B57; color:#fff !important; }
.mapcard .attr { margin:0.45rem 0 0; font-size:0.72rem; color:#56697D !important; text-align:right; }
.chart-title { color:var(--muted) !important; font-size:0.88rem; margin:0.6rem 0 0.2rem; }
.chart { width:100%; height:auto; overflow:visible; }
.chart .danger { stroke:#9FB3C6; stroke-dasharray:4 4; stroke-width:1.2; }
.chart .dlab, .chart .axis { fill:#56697D; font-size:10px; font-family:Geist, system-ui, sans-serif; }
.chart .vlab { font-size:11px; font-weight:600; font-family:Geist, system-ui, sans-serif; }
.chartsec { margin-top:2.6rem; }
.chartsec h3 { font-size:1.6rem; letter-spacing:-0.02em; margin:0 0 0.3rem; color:var(--ink) !important; }
.chartsub { color:var(--muted) !important; max-width:70ch; margin:0 0 1rem; }
.nws { position:relative; }
.nwstitle { position:absolute; z-index:1001; top:12px; left:50%; transform:translateX(-50%); background:rgba(244,244,244,.94);
  border:1px solid #9AA5B1; border-radius:8px; padding:0.45rem 1rem; text-align:center; font-weight:600;
  color:var(--ink) !important; box-shadow:0 4px 14px rgba(15,42,67,.18); white-space:nowrap; }
.nwstitle span { font-weight:400; font-size:0.85rem; color:var(--muted) !important; }
.legend { position:absolute; z-index:1001; left:14px; bottom:24px; list-style:none; margin:0; padding:0.7rem 0.9rem;
  background:rgba(244,244,244,.95); border:1px solid #9AA5B1; border-radius:8px; box-shadow:0 4px 14px rgba(15,42,67,.18);
  display:grid; grid-template-columns:repeat(2, auto); gap:0.15rem 0.9rem; font-size:0.74rem; }
.legend li { display:flex; align-items:center; gap:0.45rem; color:var(--ink) !important; }
.legend .lg-note { grid-column:1 / -1; color:var(--muted) !important; }
.sw.hatch { background:repeating-linear-gradient(45deg,#4A6278 0 3px,transparent 3px 7px) !important; }
.sw.solid { background:#4A6278 !important; }
.sw { width:18px; height:12px; border-radius:3px; flex:none; border:1px solid rgba(0,0,0,.15); }
.arrow-sw { width:22px; text-align:center; font-weight:700; color:var(--ink); }
.chartsec .attr { margin:0.45rem 0 0; font-size:0.72rem; color:#56697D !important; }
.regline { margin-top:0.8rem; }
.aqi-top { display:flex; gap:1rem; align-items:center; }
.aqi-num { flex:none; width:76px; height:76px; border-radius:50%; display:grid; place-items:center; font-size:1.7rem;
  font-weight:700; box-shadow: inset 0 0 0 3px rgba(0,0,0,.08); }
.aqi-name { font-size:1.15rem; color:var(--ink) !important; } .aqi-advice { margin:0.2rem 0 0; color:var(--muted) !important; }
.aqi-scale { position:relative; display:flex; height:10px; border-radius:99px; overflow:visible; margin:1.1rem 0 0.3rem; }
.aqi-scale span:first-child { border-radius:99px 0 0 99px; } .aqi-scale span:last-of-type { border-radius:0 99px 99px 0; }
.aqi-scale i { position:absolute; top:-5px; width:4px; height:20px; background:var(--ink); border-radius:2px; transform:translateX(-50%); }
.aqi-scale-l { position:relative; height:1rem; font-size:0.7rem; }
.aqi-scale-l span { position:absolute; transform:translateX(-50%); color:#56697D !important; }
.aqi-scale-l span:first-child { transform:none; } .aqi-scale-l span:last-child { transform:translateX(-100%); }
.aqi-main { margin:0.9rem 0 0.4rem; color:var(--muted) !important; } .aqi-main b { color:var(--ink); }
.pollutants { list-style:none; margin:0; padding:0; display:grid; grid-template-columns:1fr 1fr; gap:0.4rem; }
.pollutants li { background:var(--cream); border-radius:10px; padding:0.45rem 0.6rem; font-size:0.82rem; color:var(--muted) !important; }
.pollutants span { display:block; color:var(--muted) !important; font-size:0.75rem; } .pollutants b { font-size:1.1rem; color:var(--ink); }
.rose-row { display:grid; grid-template-columns:2.2rem 1fr 2.4rem; gap:0.6rem; align-items:center; padding:0.22rem 0; }
.rose-row span { color:var(--muted) !important; font-size:0.88rem; }
.rtrack { height:12px; background:var(--cream); border-radius:99px; overflow:hidden; }
.rtrack span { display:block; height:100%; background:#9FB3C6; border-radius:99px; }
.rose-row.now .rtrack span { background:var(--orange); } .rose-row.now span.rd, .rose-row.now span.rv { color:var(--ink) !important; font-weight:700; }
.rv { text-align:right; font-variant-numeric:tabular-nums; }
.months { display:grid; grid-template-columns:repeat(12, 1fr); gap:0.35rem; height:160px; margin-top:0.5rem; }
.mcol { display:flex; flex-direction:column; align-items:center; gap:0.25rem; }
.mbars { flex:1; width:100%; display:flex; align-items:flex-end; justify-content:center; gap:2px; }
.mbars span { width:42%; border-radius:4px 4px 0 0; } .ph, .key.ph { background:#0F2A43; } .de, .key.de { background:#C77C00; }
.mcol i { font-style:normal; font-size:0.72rem; color:#56697D; }
.key { display:inline-block; width:10px; height:10px; border-radius:2px; margin:0 0.25rem 0 0.6rem; vertical-align:middle; }
.rrmap { width:100%; border-radius:18px; overflow:hidden; background:#DDE7EF; z-index:0; }
.rrmap.close { height:clamp(320px, 38vw, 480px); }
.rrmap { position:relative; }
.rrmap-inner { position:absolute; inset:0; }
.rrmap:empty::before { content:"Loading map… If this message stays, the map script didn't run. Refresh the page.";
  display:block; padding:1rem; color:#4A6278; font-size:0.9rem; }
.rr-loading, .rr-problem { position:absolute; inset:auto 12px 12px 12px; z-index:1001; background:#fff; color:var(--ink) !important;
  border-radius:10px; padding:0.6rem 0.8rem; font-size:0.85rem; box-shadow:0 4px 14px rgba(15,42,67,.18); }
.rr-problem { padding-left:2.4rem !important; background:#fff url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 24 24' fill='none' stroke='%23D6402B' stroke-width='2' stroke-linecap='round' stroke-linejoin='round'%3E%3Cpath d='M12 3 2 20h20z'/%3E%3Cpath d='M12 10v4M12 17h.01'/%3E%3C/svg%3E") no-repeat 0.8rem center / 18px !important; }
.rrmap.region { height:clamp(420px, 58vw, 680px); }
.rr-pin { background:none; border:none; }
.rr-pin .avatar { margin:0 auto; } .rr-pin .pulse { bottom:-2px; }
.leaflet-tooltip.rr-city { font-weight:700; color:#0F2A43; background:#fff; border:none; box-shadow:0 2px 8px rgba(15,42,67,.25); }
.leaflet-container { font-family:'Geist', system-ui, sans-serif !important; }
.leaflet-control-zoom a { color:#0F2A43 !important; }
.sheet { position:static !important; margin-top:0.6rem; box-shadow:none !important; border:1px solid var(--line); }
.rose { width:100%; max-width:340px; display:block; margin:0.2rem auto; overflow:visible; }
.ring { fill:none; stroke:#D3E0EA; stroke-dasharray:3 4; }
.petal { fill:#9FB3C6; stroke:#fff; stroke-width:1.5; transform-origin:0 0; }
.petal.now { fill:var(--orange); }
.rlab { font: 500 11px Geist, system-ui, sans-serif; fill:#56697D; text-anchor:middle; }
.rlab.now { fill:var(--ink); font-weight:700; }
/* ---------- Wind arrows on the maps ---------- */
.rr-arrow { background:none; border:none; }
.wa { position:relative; width:80px; height:80px; transform:rotate(var(--rot)); transition: scale .2s ease; }
.rr-arrow:hover .wa { scale:1.3; }
.wa.s1 { color:#5C7A96; } .wa.s2 { color:#0F2A43; } .wa.s3 { color:#C77C00; } .wa.s4 { color:#D6402B; }
.wa-line { position:absolute; left:50%; top:50%; width:4px; height:var(--len); margin-left:-2px;
  transform:translateY(-50%); border-radius:2px; background:currentColor; opacity:.55; }
.wa-head { position:absolute; left:50%; top:calc(50% - var(--len) / 2 - 10px); margin-left:-7px; width:0; height:0;
  border-left:7px solid transparent; border-right:7px solid transparent; border-bottom:12px solid currentColor; }
.wa-dot { position:absolute; left:50%; top:calc(50% + var(--len) / 2 - 4px); width:9px; height:9px; margin-left:-4.5px;
  border-radius:50%; background:#fff; box-shadow:0 0 0 2.5px currentColor; animation: glide var(--dur) linear infinite; }
@keyframes glide { 0% { transform:translateY(0); opacity:0; } 15%, 80% { opacity:1; }
  100% { transform:translateY(calc(-1 * var(--len))); opacity:0; } }
.arrow-key { width:18px; text-align:center; font-weight:700; color:var(--ink); }
.rr-flow { animation: flow 1.2s linear infinite; }
@keyframes flow { to { stroke-dashoffset:-24; } }
/* ---------- Hover: things respond when you point at them ---------- */
.card, .call-side { transition: transform .25s var(--ease), box-shadow .25s ease; }
.card:hover, .call-side:hover { transform: translateY(-4px); box-shadow: var(--shadow-lg); }
.board { transition: transform .25s var(--ease), box-shadow .25s ease; }
.board:hover { transform: translateY(-4px); box-shadow: var(--shadow-xl); }
.meter:hover { background: rgba(255,255,255,.07); padding-inline:0.4rem; }
.meter:hover .fill { filter: brightness(1.25); }
.petal { transition: scale .2s ease, filter .2s ease; cursor:pointer; }
.petal:hover { scale: 1.08; filter: brightness(1.08) drop-shadow(0 4px 6px rgba(15,42,67,.25)); }
.srow { transition: transform .2s ease, box-shadow .2s ease; }
.srow:hover { transform: translateX(6px); box-shadow: 0 8px 20px rgba(15,42,67,.1); }
.pollutants li { transition: transform .2s ease, background .2s ease; }
.pollutants li:hover { transform: translateY(-2px); background:#DCE9F4; }
.legend li { transition: transform .15s ease; } .legend li:hover { transform: translateX(3px); }
#go:hover { transform: translateY(-2px); box-shadow: 0 10px 22px rgba(245,183,0,.45); }
.rr-pin .avatar { transition: scale .2s ease; } .rr-pin:hover .avatar { scale: 1.12; }
.navlinks a { background: linear-gradient(var(--orange), var(--orange)) no-repeat 0 100% / 0% 2px; transition: background-size .25s ease; padding-bottom:2px; }
.navlinks a:hover { background-size: 100% 2px; color: var(--ink) !important; }
.aqi-num.alert { animation: breathe 2.4s ease-in-out infinite; }
@keyframes breathe { 50% { box-shadow: inset 0 0 0 3px rgba(0,0,0,.08), 0 0 0 10px rgba(255,126,0,.18); } }
.meter { animation: rise .45s var(--ease) backwards; }
.meter:nth-child(2) { animation-delay:.03s; } .meter:nth-child(3) { animation-delay:.06s; } .meter:nth-child(4) { animation-delay:.09s; }
.meter:nth-child(5) { animation-delay:.12s; } .meter:nth-child(6) { animation-delay:.15s; } .meter:nth-child(7) { animation-delay:.18s; }
.meter:nth-child(8) { animation-delay:.21s; } .meter:nth-child(9) { animation-delay:.24s; } .meter:nth-child(10) { animation-delay:.27s; }
/* ---------- Floating layers are frosted glass (they sit over the map) ---------- */
.chip, .nwstitle, .legend { background:var(--glass) !important; -webkit-backdrop-filter: blur(16px) saturate(180%);
  backdrop-filter: blur(16px) saturate(180%); border:1px solid rgba(255,255,255,.65) !important;
  box-shadow: 0 10px 30px -10px rgba(15,42,67,.35) !important; }
.chip { display:inline-flex; align-items:center; gap:0.4rem; }
.nav { position:sticky; top:0; z-index:1100; margin:0 -1rem; padding:0.7rem 1rem !important; border-radius:0 0 16px 16px;
  background:rgba(234,242,248,.72); -webkit-backdrop-filter: blur(18px) saturate(180%); backdrop-filter: blur(18px) saturate(180%); }
/* ---------- Icons ---------- */
.ico { flex:none; vertical-align:-3px; }
.ico, .ico * { fill:none !important; color:inherit !important; }
.meter .ico, .meter .ico * { stroke:#9FB3C6 !important; }
.meter.hot .ico, .meter.hot .ico * { stroke:var(--orange) !important; }
.chip .ico, .chip .ico *, .arrow-key .ico, .arrow-key .ico * { stroke:var(--ink) !important; }
.meter .ico { color:#9FB3C6 !important; } .meter.hot .ico { color:var(--orange) !important; }
.chip .ico, .arrow-key .ico { color:var(--ink) !important; }
.mname { display:inline-flex; align-items:center; gap:0.45rem; }
.arrow-key { display:inline-flex; width:18px; justify-content:center; color:var(--ink); }
.rr-pin .avatar svg, .up-dot svg { display:block; }
/* ---------- Instant press feedback (on press, not on release) ---------- */
#go:active, .meter:active, .petal:active, .srow:active, .pollutants li:active { transform: scale(.97); transition-duration: 80ms; }
.petal:active { scale: .97; }
/* ---------- Browser surfaces in our colors ---------- */
::selection { background:#F5B700; color:#0F2A43; }
:focus-visible { outline:3px solid #F5B700 !important; outline-offset:2px; border-radius:8px; }
html { scrollbar-color:#9FB3C6 #EAF2F8; }
input, textarea { caret-color:#0F2A43; }
.srow b, .pct, .aqi-num, .pollutants b, .rv, .rlab { font-variant-numeric: tabular-nums; }
/* ---------- Type: tracking changes with size ---------- */
.why b { font-weight:700; color:var(--ink); }
.attr, .chart-title, .board-head, .legend { letter-spacing:.01em; }
.call { letter-spacing:-0.035em !important; }
/* ---------- Respect device settings ---------- */
@media (prefers-reduced-transparency: reduce) {
  .chip, .nwstitle, .legend, .nav { background:#fff !important; backdrop-filter:none !important; -webkit-backdrop-filter:none !important; }
}
@media (prefers-contrast: more) {
  .card, .call-side, .chip, .nwstitle, .legend { border:2px solid var(--ink) !important; }
  .attr, .chart-title, .why { color:var(--ink) !important; }
}
/* ---------- Motion: one entrance for the page, then small moves that show what changed ---------- */
@keyframes rise { from { opacity:0; transform:translateY(10px); } }
@keyframes grow { from { transform:scaleX(0); } }
@keyframes bloom { from { transform:scale(0); } }
@keyframes draw { from { stroke-dashoffset:1; } }
@keyframes slide { from { left:0%; } }
.hero h1, .hero .lede { animation: rise .7s var(--ease) both; }
.hero .lede { animation-delay:.12s; }
.call-side .call, .call-side .why { animation: rise .6s var(--ease) backwards; } .call-side .why { animation-delay:.08s; }
.fill { transform-origin:left; animation: grow .9s var(--ease) backwards; }
.petal { animation: bloom .7s var(--ease) backwards; }
.chart .draw { stroke-dasharray:1; animation: draw 1.1s ease-out both; }
.aqi-scale i { animation: slide .9s var(--ease) both; }
.call.indoor span { background-size:0% 100%; background-repeat:no-repeat; animation: hl .7s .25s ease-out forwards; }
@keyframes hl { to { background-size:100% 100%; } }
#go { transition: transform .15s ease, background .15s ease; } #go:active { transform: scale(.97); }
@keyframes fade { from { opacity:0; } }
@media (prefers-reduced-motion: reduce) {
  *, *::before, *::after { animation:none !important; transition:none !important; }
  .call-side, .board { animation: fade .2s ease-out !important; }
  .call.indoor span { background-size:100% 100%; }
}
.scores, .how { margin-top:2.6rem; }
.scores h3, .how h3 { font-size:1.6rem; letter-spacing:-0.02em; margin:0 0 0.3rem; color:var(--ink) !important; }
.scores > p, .how p { color:var(--muted) !important; max-width:62ch; line-height:1.55; }
.srow { display:grid; grid-template-columns: 1.6fr 1fr 1fr 1fr; padding:1rem 1.2rem; border-radius:14px; margin-top:0.6rem;
  align-items:baseline; }
.srow.ours { background:var(--ink); } .srow.ours, .srow.ours span { color:var(--cream) !important; } .srow.ours b { color:var(--orange) !important; }
.srow.base { background:var(--paper); border:1px solid var(--line); } .srow.base span, .srow.base b { color:var(--ink) !important; }
.srow b { font-size:1.5rem; letter-spacing:-0.02em; } .sname { font-weight:600; }
.fine { color:var(--muted) !important; font-size:0.88rem; margin-top:2.4rem; border-top:1px solid var(--line); padding-top:1rem; }
/* Tablets and small laptops */
@media (max-width: 1080px) {
  .result { grid-template-columns: 1fr 1fr; } .panels { grid-template-columns: 1fr 1fr; }
  .meter { grid-template-columns: 1fr 2.8rem; } .meter .track { grid-column: 1 / -1; grid-row: 2; }
}
/* Phones and narrow tablets: everything in one column */
@media (max-width: 820px) {
  .result, .panels { grid-template-columns: 1fr; } .navlinks { display:none; }
  #picker { flex-direction: column !important; align-items: stretch !important; }
  #picker > * { width: 100% !important; min-width: 0 !important; }
  .srow { grid-template-columns: 1fr 1fr 1fr; } .sname { grid-column: 1 / -1; margin-bottom:0.3rem; }
  .sheet { position:static; margin-top:0.6rem; box-shadow:none; border:1px solid var(--line); }
  .legend { position:static !important; margin-top:0.6rem; grid-template-columns:1fr 1fr; }
  .nwstitle { position:static; transform:none; white-space:normal; font-size:0.85rem; margin-bottom:0.5rem; box-shadow:none; }
  .nws { display:flex; flex-direction:column; } .nws .nwstitle { order:-1; }
}
/* Small phones */
@media (max-width: 480px) {
  .call-side, .card { padding: 1.1rem; border-radius: 16px; } .board { padding: 1.1rem; }
  .mapcard { padding: 0.8rem; } .chip { font-size:0.75rem; } .sheet-row { flex-wrap:wrap; } .srow { padding: 0.8rem 0.9rem; } .srow b { font-size: 1.25rem; }
  .nav { padding-bottom: 1rem; }
}
/* Big monitors */
@media (min-width: 1600px) {
  .result { grid-template-columns: 1fr 1.2fr; }
}
"""

# Put the map library (Leaflet) and our map script straight into the page.
# (Serving them as separate files didn't work in every setup, so we read the files and paste them in.)
STATIC = DATA_DIR.parent / "project" / "static"
def read_static(name):
    return (STATIC / name).read_text(encoding="utf-8")

MAP_HEAD = ("<style>" + read_static("leaflet/leaflet.css") + "</style>"
            "<script>" + read_static("leaflet/leaflet.js") + "</script>"
            "<script>" + read_static("recess_maps.js") + "</script>")

HEAD = MAP_HEAD + ('<link rel="preconnect" href="https://fonts.googleapis.com">'
        '<link href="https://fonts.googleapis.com/css2?family=Geist:wght@400;500;600;700'
        '&family=Instrument+Serif:ital@1&display=swap" rel="stylesheet">')

# Always show the light (cream) version, even if the computer is in dark mode
FORCE_LIGHT = "() => { document.body.classList.remove('dark'); }"

with gr.Blocks(title="Recess Ready: Philadelphia") as demo:
    gr.HTML(HERO)
    with gr.Row(elem_id="picker"):
        date_box = gr.Dropdown(test_dates, value=test_dates[0], label="Pick a school day from the 2025-26 test year", scale=3)
        go = gr.Button("Make the call", elem_id="go", scale=1)
    result = gr.HTML(recess_check(test_dates[0]))
    gr.HTML(scores_html())
    go.click(recess_check, inputs=date_box, outputs=result)
    date_box.change(recess_check, inputs=date_box, outputs=result)

demo.launch(css=CSS, head=HEAD, js=FORCE_LIGHT)
